<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/50_build_app_artefacts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# %% [markdown]
# Build clean app artefacts from extracted runs — strategy ≠ model
# ---------------------------------------------------------------
# Goal:
# - 'strategy' in output = **trading rule** only (what the app should list).
# - Model architecture/feature/window/fold go to separate metadata columns.
# - If several models share the same rule, keep the best Sharpe for the app list.
#
# Inputs:
#   /content/drive/MyDrive/gt-markets/app-demo/extracted/{daily|weekly}/*.csv
#     must contain Date and one of: position | prob_up | y_pred / signal
#     may NOT contain Close -> fetch from data/processed or baseline signals.
#
# Outputs for the app (per asset):
#   - metrics_keywords_D.csv / metrics_keywords_W.csv (strategy-level, best Sharpe)
#   - signals_<STRATEGY>_D.csv / signals_<STRATEGY>_W.csv (time series)
#   - figs/<STRATEGY>_<FREQ>.png (quick sanity plot)
#
# Notes:
# - Trading rules inferred from file/content:
#     • If a binary position/threshold from prob_up→(>=0.5):  R_PROB50
#     • If filename mentions wNN (e.g., w30) as a step/rolling rule: R_WNN
#     • Else, fallback: R_GENERIC
# - Model metadata parsed separately: model, features (RAW/ENG), variant (BASE/EXT),
#   window (wNN if present), fold (fK), dataset (val/test/train).
#
# After build:
#   Push AppDemo/artefacts to Git and reload Streamlit.

# %%
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re, math

# --- Paths (keep consistent with repo)
PROJECT_DIR    = Path("/content/drive/MyDrive/gt-markets")
SRC_EXTRACT    = PROJECT_DIR / "app-demo" / "extracted"   # daily/weekly folders
ARTE_ROOT      = PROJECT_DIR / "AppDemo" / "artefacts"
DATA_PROCESSED = PROJECT_DIR / "data" / "processed"

ASSETS        = ["GOLD","BTC","OIL","USDCNY"]
FREQ_DIRS     = {"daily":"D", "weekly":"W"}
ANNUALIZATION = {"D":252, "W":52}
COST_BPS      = 5

for a in ASSETS:
    (ARTE_ROOT/a/"figs").mkdir(parents=True, exist_ok=True)

# -----------------------
# Helper functions
# -----------------------
def _infer_asset(text: str):
    t = text.lower()
    if ("gold" in t) or ("gc=f" in t): return "GOLD"
    if ("btc" in t)  or ("bitcoin" in t): return "BTC"
    if ("oil" in t)  or ("cl=f" in t): return "OIL"
    if "usdcny" in t: return "USDCNY"
    return None

def _infer_model(text: str):
    t = text.lower()
    if "xgb" in t or "xgboost" in t: return "XGB"
    if "lstm" in t: return "LSTM"
    if "gru" in t:  return "GRU"
    if "rf" in t:   return "RF"
    if re.search(r"(^|[^a-z])lr($|[^a-z])", t): return "LR"
    if "mlp" in t: return "MLP"
    if "svm" in t: return "SVM"
    return "MODEL"

def _infer_features(text: str):
    t = text.lower()
    # RAW/ENG + BASE/EXT flags
    src = "RAW" if "raw" in t else ("ENG" if "eng" in t else "UNK")
    var = "EXT" if "ext" in t else ("BASE" if "base" in t else "UNK")
    return f"{src}_{var}"

def _infer_window(text: str):
    m = re.search(r"w(\d{1,3})", text.lower())
    return f"w{m.group(1)}" if m else None

def _infer_fold(text: str):
    m = re.search(r"f(\d+)", text.lower())
    return f"f{m.group(1)}" if m else None

def _infer_dataset(text: str):
    t = text.lower()
    if "val" in t:  return "val"
    if "test" in t: return "test"
    if "train" in t:return "train"
    return "unk"

def _detect_positions(df: pd.DataFrame):
    # Returns binary 0/1 series AND a rule label describing the trading strategy
    lower = {c.lower(): c for c in df.columns}
    # Direct position (already 0/1)
    if "position" in lower:
        s = pd.to_numeric(df[lower["position"]], errors="coerce").fillna(0.0)
        pos = (s >= 0.5).astype(int)
        return pos, "R_GENERIC"
    # Probabilities -> threshold rule at 0.5
    for cand in ["prob_up","proba_up","p_up","prob","y_pred_proba","pred_proba","probability"]:
        if cand in lower:
            s = pd.to_numeric(df[lower[cand]], errors="coerce").fillna(0.0)
            pos = (s >= 0.5).astype(int)
            return pos, "R_PROB50"
    # Signed predictions -> long if >0
    for cand in ["signal","y_pred","pred"]:
        if cand in lower:
            s = pd.to_numeric(df[lower[cand]], errors="coerce").fillna(0.0)
            pos = (s > 0).astype(int)
            return pos, "R_GENERIC"
    return None, None

def _maybe_rule_from_filename(text: str, existing_rule: str):
    # If the filename carries a wNN hint and we expect a step/rolling rule, tag it.
    w = _infer_window(text)
    if w:
        return f"R_{w.upper()}"
    return existing_rule or "R_GENERIC"

def _load_processed_close(asset: str, F: str):
    candidates = [
        DATA_PROCESSED / f"{asset.lower()}_{F}.csv",
        DATA_PROCESSED / f"{asset}_{F}.csv",
        DATA_PROCESSED / f"prices_{asset}_{F}.csv",
        DATA_PROCESSED / f"{asset.lower()}_{F.lower()}.csv",
    ]
    for p in candidates:
        if not p.exists(): continue
        try:
            d = pd.read_csv(p)
            if "Date" not in d.columns: continue
            close_col = next((c for c in ["Close","Adj Close","close","adjclose","price","Price"] if c in d.columns), None)
            if not close_col: continue
            s = pd.Series(pd.to_numeric(d[close_col], errors="coerce").values,
                          index=pd.to_datetime(d["Date"], errors="coerce"), name="Close").dropna()
            return s.sort_index()
        except:  # keep going
            pass
    return None

def _fallback_close_from_signals(asset: str, F: str):
    base_dir = ARTE_ROOT / asset
    if not base_dir.exists(): return None
    for p in sorted(base_dir.glob(f"signals_*_{F}.csv")):
        try:
            d = pd.read_csv(p)
            if {"Date","Close"}.issubset(d.columns):
                s = pd.Series(pd.to_numeric(d["Close"], errors="coerce").values,
                              index=pd.to_datetime(d["Date"], errors="coerce"), name="Close").dropna()
                return s.sort_index()
        except:
            continue
    return None

def _get_close(asset: str, F: str):
    s = _load_processed_close(asset, F)
    return s if s is not None else _fallback_close_from_signals(asset, F)

def _to_equity(close: pd.Series, pos01: pd.Series, F: str, cost_bps: float):
    close = close.copy().dropna()
    close.index = pd.to_datetime(close.index)
    close = close.sort_index()

    pos = pos01.copy().astype(float)
    pos.index = pd.to_datetime(pos.index)
    pos = pos.sort_index().reindex(close.index).ffill().fillna(0.0).clip(0,1).astype(int)

    ret = close.pct_change(fill_method=None).fillna(0.0)
    trans = pos.diff().abs().fillna(0.0)
    strat = (pos.shift(1).fillna(0.0) * ret) - trans * (cost_bps/1e4)

    eq = (1 + strat).cumprod().replace([np.inf,-np.inf], np.nan).fillna(1.0)
    ann = ANNUALIZATION[F]
    mu  = strat.mean() * ann
    sd0 = strat.std(ddof=0); sd = (sd0 * (ann**0.5)) if sd0 > 0 else np.nan
    shr = mu/sd if (sd and sd > 0) else np.nan
    mdd = (eq/eq.cummax()-1).min()
    return eq, {"Return_Ann": mu, "Vol_Ann": sd, "Sharpe": shr, "MaxDD": mdd}

# -----------------------
# Build
# -----------------------
rows = []          # per-file metrics (with model + rule)
signals_paths = [] # produced signal files

for subdir, F in FREQ_DIRS.items():
    src = SRC_EXTRACT / subdir
    if not src.exists():
        print(f"[skip] {src} not found")
        continue

    for csv in sorted(src.rglob("*.csv"), key=lambda p: p.stat().st_mtime):
        name = csv.name.lower()
        if any(x in name for x in ["leaderboard","scores","eval","backtest_summary"]):
            continue
        try:
            df = pd.read_csv(csv)
        except Exception as e:
            print(f"[warn] read {csv.name} failed: {e}")
            continue

        # Date
        if "Date" not in df.columns:
            if "date" in df.columns: df = df.rename(columns={"date":"Date"})
            else: continue
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date"])

        # Asset
        asset = _infer_asset(csv.as_posix())
        if not asset:
            # optional: peek into a column
            for c in ["asset","symbol","ticker"]:
                if c in df.columns:
                    asset = _infer_asset(str(df[c].iloc[0]))
                    if asset: break
        if asset not in ASSETS:
            continue

        # Trading positions + rule
        pos01, rule = _detect_positions(df)
        if pos01 is None:
            continue
        pos01.index = df["Date"]
        rule = _maybe_rule_from_filename(csv.name, rule)  # prefer wNN tag if present
        strategy = rule  # <-- IMPORTANT: strategy used by the app = trading rule ONLY

        # Model meta (kept separate from strategy)
        model    = _infer_model(csv.name)
        feats    = _infer_features(csv.as_posix())
        window   = _infer_window(csv.name)
        fold     = _infer_fold(csv.name)
        dataset  = _infer_dataset(csv.name)

        # Close series
        if "Close" in df.columns:
            close = pd.Series(pd.to_numeric(df["Close"], errors="coerce").values,
                              index=df["Date"], name="Close")
        else:
            close = _get_close(asset, F)
            if close is None:
                print(f"[warn] {csv.name}: no Close available; skipped")
                continue

        # Compute metrics
        eq, mets = _to_equity(close, pos01, F, COST_BPS)

        # Save signals_<strategy>_<F>.csv
        out_dir = ARTE_ROOT / asset
        pos_a = pos01.reindex(eq.index).ffill().fillna(0).astype(int)
        trans = pos_a.diff().fillna(0).abs().astype(int)
        signal = trans.where(trans > 0, np.nan)
        signal = signal.where(signal.isna(), pos_a)

        sig_df = pd.DataFrame({
            "Date": eq.index.strftime("%Y-%m-%d"),
            "signal": signal.values,
            "position": pos_a.values,
            "Close": close.reindex(eq.index).ffill().values,
            "strategy": strategy,       # trading rule only
            "model": model,             # model meta (kept separate)
            "features": feats,
            "window": window if window else "",
            "fold": fold if fold else "",
            "dataset": dataset
        })
        sig_path = out_dir / f"signals_{strategy}_{F}.csv"
        sig_df.to_csv(sig_path, index=False)
        signals_paths.append(sig_path)

        # Small plot
        fig_path = out_dir / "figs" / f"{strategy}_{F}.png"
        ax = eq.plot(figsize=(6,3), title=f"{asset} – {strategy} – {F}")
        ax.grid(alpha=0.3)
        ax.get_figure().savefig(fig_path, dpi=150, bbox_inches="tight")
        plt.close(ax.get_figure())

        # Metrics row (retain model meta for tie-break/inspection)
        rows.append({
            "asset": asset, "freq": F, "strategy": strategy,
            "model": model, "features": feats, "window": window or "",
            "fold": fold or "", "dataset": dataset,
            **mets
        })

# Aggregate → best Sharpe per (asset,freq,strategy)
if not rows:
    raise RuntimeError("No strategies built. Check extracted/ inputs and processed prices.")

dfm = pd.DataFrame(rows)
best = (
    dfm.sort_values(["asset","freq","strategy","Sharpe"], ascending=[True,True,True,False])
        .drop_duplicates(subset=["asset","freq","strategy"], keep="first")
        .reset_index(drop=True)
)
# Persist metrics_keywords_{F}.csv per asset with best Sharpe strategy rows
for a in ASSETS:
    for F in ["D","W"]:
        sub = best[(best.asset==a) & (best.freq==F)].copy()
        out_csv = ARTE_ROOT / a / f"metrics_keywords_{F}.csv"
        if len(sub)==0:
            if out_csv.exists(): out_csv.unlink(missing_ok=True)
            continue
        # keep model meta as extra columns; the app can ignore or show them
        cols = ["asset","freq","strategy","Return_Ann","Vol_Ann","Sharpe","MaxDD",
                "model","features","window","fold","dataset"]
        sub[cols].to_csv(out_csv, index=False)

print("✅ Built artefacts to:", ARTE_ROOT)

# Quick integrity check for the app (presence only)
_required = {
    "GOLD":  ["metrics_baseline_D.csv","metrics_baseline_W.csv","metrics_keywords_D.csv","metrics_keywords_W.csv"],
    "BTC":   ["metrics_baseline_D.csv","metrics_baseline_W.csv","metrics_keywords_D.csv","metrics_keywords_W.csv"],
    "OIL":   ["metrics_baseline_D.csv","metrics_baseline_W.csv","metrics_keywords_D.csv","metrics_keywords_W.csv"],
    "USDCNY":["metrics_baseline_D.csv","metrics_baseline_W.csv","metrics_keywords_D.csv","metrics_keywords_W.csv"],
}
missing = []
for asset, files in _required.items():
    for f in files:
        if not (ARTE_ROOT/asset/f).exists():
            missing.append(f"{asset}/{f}")
if missing:
    print("⚠️ Missing artefact files:")
    for m in missing: print("  -", m)
else:
    print("✅ All required files present for the demo app.")


Mounted at /content/drive
✅ Built artefacts to: /content/drive/MyDrive/gt-markets/AppDemo/artefacts
✅ All required files present for the demo app.
